# Phase 4: Activity/Itinerary & Budget/Currency Agents

Complete the 5-agent pipeline with real LLM-powered agents:
- **Activity/Itinerary Agent** — day-by-day planning with weather-appropriate activities
- **Budget/Currency Agent** — itemized cost breakdown with currency conversion

### Tools introduced:
- `search_activities` — destination-specific activities organized by type (cultural, adventure, relaxation, food)
- `get_weather_forecast` — climate-aware weather forecast (tropical/mediterranean/temperate)
- `convert_currency` — USD to local currency conversion (40+ destinations)
- `calculate_budget` — comprehensive budget breakdown with region-based food/transport estimates

## 1. Setup

In [ ]:
import sys
import os
import json

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"LLM Provider: {os.getenv('LLM_PROVIDER', 'anthropic')}")

## 2. Test Activity Tools

In [ ]:
from src.tools.activity_tools import search_activities, get_weather_forecast

# Test activity search for known destination
print("=== Activities: Bali (beach, culture) ===")
result = json.loads(search_activities.invoke({"destination": "Bali", "preferences": "beach, culture", "num_days": 5}))
for cat, items in result["activities_by_type"].items():
    print(f"  {cat} ({len(items)} activities):")
    for a in items:
        print(f"    - {a['name']} (${a['cost']}, {a['duration']})")
print()

# Test for dynamic destination
print("=== Activities: Santorini (romantic) - dynamic ===")
result = json.loads(search_activities.invoke({"destination": "Santorini", "preferences": "romantic", "num_days": 7}))
for cat, items in result["activities_by_type"].items():
    print(f"  {cat} ({len(items)} activities):")
    for a in items:
        print(f"    - {a['name']} (${a['cost']}, {a['duration']})")

In [ ]:
# Test weather forecasts
print("=== Weather: Bali (March - dry season) ===")
result = json.loads(get_weather_forecast.invoke({"destination": "Bali", "start_date": "2025-03-15", "num_days": 5}))
print(f"Climate: {result['climate']}, Season: {result['season']}")
for d in result['forecast']:
    outdoor = 'Yes' if d['outdoor_friendly'] else 'No'
    print(f"  Day {d['day']}: {d['description']:30s} | {d['temperature_c']}C ({d['temperature_f']}F) | Rain: {d['rain_probability_pct']}% | Outdoor: {outdoor}")
print()

print("=== Weather: Paris (December - wet season) ===")
result = json.loads(get_weather_forecast.invoke({"destination": "Paris", "start_date": "2025-12-15", "num_days": 5}))
print(f"Climate: {result['climate']}, Season: {result['season']}")
for d in result['forecast']:
    outdoor = 'Yes' if d['outdoor_friendly'] else 'No'
    print(f"  Day {d['day']}: {d['description']:30s} | {d['temperature_c']}C ({d['temperature_f']}F) | Rain: {d['rain_probability_pct']}% | Outdoor: {outdoor}")

## 3. Test Budget Tools

In [ ]:
from src.tools.budget_tools import convert_currency, calculate_budget

# Test currency conversion
print("=== Currency Conversions ===")
for dest in ["Bali", "Tokyo", "Paris", "Maldives", "Cancun"]:
    result = json.loads(convert_currency.invoke({"amount_usd": 1000, "destination": dest}))
    print(f"  $1000 USD -> {result['formatted']} ({result['local_currency']})")
print()

# Test budget calculator
print("=== Budget: Bali (2 pax, 5 days, $3000 budget) ===")
result = json.loads(calculate_budget.invoke({
    "budget_usd": 3000, "travelers": 2, "num_days": 5,
    "flight_cost": 1900, "hotel_cost": 460, "activity_cost": 380,
    "destination": "Bali",
}))
print(f"Status: {result['budget_status']} | Total: ${result['total_usd']} | Budget: ${result['budget_usd']}")
print(f"Per person: ${result['per_person_cost']} | Daily spending: ${result['daily_spending']}")
for k, v in result['breakdown'].items():
    print(f"  {k:20s}: ${v}")
conv = result['currency_conversion']
print(f"Local: {conv['total_local_currency']:,.0f} {conv['currency_code']}")
if result['savings_tips']:
    print("Savings tips:")
    for tip in result['savings_tips']:
        print(f"  - {tip}")

## 4. Test Activity Agent (ReAct with Tools)

In [ ]:
from src.agents.activity_agent import _run_react_agent as activity_react

print("=== Activity Agent: Bali (5 days, beach + culture) ===")
print()
result = activity_react("Bali", ["beach", "culture"], "2025-03-15", 5)
print(result[:2000] if len(result) > 2000 else result)

## 5. Test Budget Agent (ReAct with Tools)

In [ ]:
from src.agents.budget_agent import _run_react_agent as budget_react

print("=== Budget Agent: Bali ($3000, 2 pax, 5 days) ===")
print()
result = budget_react("Bali", 3000, 2, 5, 1900, 460, 380)
print(result[:2000] if len(result) > 2000 else result)

## 6. Full 5-Agent Pipeline

All 5 agents are now real (LLM + Tools). The complete pipeline:
1. Intent Parser -> 2. Destination Research -> 3. Flight Search -> 4. Hotel Search -> 5. Activity/Itinerary -> 6. Budget/Currency -> 7. Aggregator

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from src.state import TravelPlanState
from src.supervisor import parse_intent, supervisor_router, aggregate_results
from src.agents.destination_agent import destination_research_agent
from src.agents.flight_agent import flight_search_agent
from src.agents.hotel_agent import hotel_search_agent
from src.agents.activity_agent import activity_itinerary_agent
from src.agents.budget_agent import budget_currency_agent

builder = StateGraph(TravelPlanState)
builder.add_node("intent_parser", parse_intent)
builder.add_node("destination_research", destination_research_agent)
builder.add_node("flight_search", flight_search_agent)
builder.add_node("hotel_search", hotel_search_agent)
builder.add_node("activity_itinerary", activity_itinerary_agent)
builder.add_node("budget_currency", budget_currency_agent)
builder.add_node("aggregator", aggregate_results)

builder.add_edge(START, "intent_parser")
builder.add_conditional_edges("intent_parser", supervisor_router)
builder.add_conditional_edges("destination_research", supervisor_router)
builder.add_conditional_edges("flight_search", supervisor_router)
builder.add_conditional_edges("hotel_search", supervisor_router)
builder.add_conditional_edges("activity_itinerary", supervisor_router)
builder.add_conditional_edges("budget_currency", supervisor_router)
builder.add_edge("aggregator", END)

graph = builder.compile()
print("Graph compiled with ALL 5 REAL agents!")
print()

print("Running full 5-agent pipeline for Maldives...")
print()
for step in graph.stream({
    "messages": [
        HumanMessage(
            content="Plan a 7-day luxury honeymoon to the Maldives for 2 people, "
            "budget $8000, departing from Chicago. Dates: 2025-04-10 to 2025-04-17. "
            "Preferences: snorkeling, spa, romantic dinners."
        )
    ],
}):
    node_name = list(step.keys())[0]
    node_output = step[node_name]

    label = '(REAL - LLM + Tools)' if node_name not in ('intent_parser', 'aggregator') else ''
    print(f"--- {node_name} {label} ---")
    if 'messages' in node_output and node_output['messages']:
        content = node_output['messages'][-1].content
        if len(content) > 500:
            print(f"  {content[:500]}...")
        else:
            print(f"  {content}")
    print()

## 7. Inspect Full State

Run the Maldives request and inspect all structured data from every agent.

In [ ]:
final = graph.invoke({
    "messages": [
        HumanMessage(
            content="Plan a 7-day luxury honeymoon to the Maldives for 2 people, "
            "budget $8000, departing from Chicago. Dates: 2025-04-10 to 2025-04-17. "
            "Preferences: snorkeling, spa, romantic dinners."
        )
    ],
})

print("=== FLIGHT OPTIONS ===")
flights = final.get("flight_options", {})
for i, f in enumerate(flights.get('options', []), 1):
    print(f"  {i}. {f['airline']} ({f['route']}) - ${f['total_price']} | {f['duration']}")
print()

print("=== HOTEL OPTIONS ===")
hotels = final.get("hotel_options", {})
for i, h in enumerate(hotels.get('options', []), 1):
    print(f"  {i}. {h['name']} ({h['star_rating']}*) - ${h['total_price']}/stay | {h['location']}")
print()

print("=== ITINERARY ===")
itin = final.get("itinerary", {})
print(f"Days: {itin.get('total_days')} | Activity cost: ${itin.get('total_activity_cost')} | Source: {itin.get('source')}")
for day in itin.get('days', []):
    print(f"  Day {day['day']}: {day['theme']} ({day['weather']})")
    print(f"    AM: {day['morning']['activity']} (${day['morning']['cost']})")
    print(f"    PM: {day['afternoon']['activity']} (${day['afternoon']['cost']})")
    print(f"    EVE: {day['evening']['activity']} (${day['evening']['cost']})")
print()

print("=== BUDGET SUMMARY ===")
budget = final.get("budget_summary", {})
print(f"Status: {budget.get('budget_status')} | Total: ${budget.get('total_usd')} | Budget: ${budget.get('budget_usd')}")
print(f"Per person: ${budget.get('per_person_cost')} | Source: {budget.get('source')}")
for key in ['flights', 'accommodation', 'activities', 'food_and_dining', 'local_transport', 'miscellaneous']:
    print(f"  {key:20s}: ${budget.get(key, 0)}")
conv = budget.get('currency_conversion', {})
if conv:
    print(f"  Local: {conv.get('total_local_currency', 0):,.0f} {conv.get('currency_code', '')}")
if budget.get('savings_tips'):
    print("  Savings tips:")
    for tip in budget['savings_tips']:
        print(f"    - {tip}")

## 8. Verification Checks

Verify budget math, weather-appropriate activities, and currency conversion.

In [ ]:
budget = final.get("budget_summary", {})
itin = final.get("itinerary", {})

# Budget math verification
print("=== BUDGET MATH VERIFICATION ===")
components = ['flights', 'accommodation', 'activities', 'food_and_dining', 'local_transport', 'miscellaneous']
computed_total = sum(budget.get(k, 0) for k in components)
reported_total = budget.get('total_usd', 0)
match = abs(computed_total - reported_total) < 1
print(f"  Sum of components: ${computed_total}")
print(f"  Reported total:    ${reported_total}")
print(f"  Math check:        {'PASS' if match else 'FAIL'}")
print()

# Budget status verification
budget_usd = budget.get('budget_usd', 0)
status = budget.get('budget_status', '')
if reported_total <= budget_usd * 0.95:
    expected = 'WITHIN BUDGET'
elif reported_total <= budget_usd * 1.05:
    expected = 'ON TARGET'
else:
    expected = 'OVER BUDGET'
print(f"  Budget status:  {status}")
print(f"  Expected:       {expected}")
print(f"  Status check:   {'PASS' if status == expected else 'FAIL'}")
print()

# Currency conversion verification
print("=== CURRENCY CONVERSION VERIFICATION ===")
conv = budget.get('currency_conversion', {})
if conv:
    rate = conv.get('exchange_rate', 1)
    expected_local = round(reported_total * rate, 2)
    reported_local = conv.get('total_local_currency', 0)
    match_fx = abs(expected_local - reported_local) < 1
    print(f"  Rate: 1 USD = {rate} {conv.get('currency_code', '')}")
    print(f"  Expected local: {expected_local:,.2f}")
    print(f"  Reported local: {reported_local:,.2f}")
    print(f"  FX check:       {'PASS' if match_fx else 'FAIL'}")
print()

# Itinerary day count
print("=== ITINERARY VERIFICATION ===")
days = itin.get('days', [])
print(f"  Days planned: {len(days)}")
print(f"  Total activity cost: ${itin.get('total_activity_cost', 0)}")
computed_act_cost = sum(
    d['morning']['cost'] + d['afternoon']['cost'] + d['evening']['cost']
    for d in days
)
print(f"  Computed from days:  ${computed_act_cost}")
print(f"  Activity cost check: {'PASS' if computed_act_cost == itin.get('total_activity_cost', -1) else 'FAIL'}")

## Summary

Phase 4 complete — all 5 agents are now real LLM-powered ReAct agents:

| Agent | Tools | Status |
|-------|-------|--------|
| Destination Research | web_search, get_destination_info | Real (Phase 2) |
| Flight Search | search_flights, get_airport_code | Real (Phase 3) |
| Hotel Search | search_hotels, get_hotel_reviews | Real (Phase 3) |
| Activity/Itinerary | search_activities, get_weather_forecast | Real (Phase 4) |
| Budget/Currency | calculate_budget, convert_currency | Real (Phase 4) |

### Key features:
- Weather-aware itinerary planning (indoor activities on rainy days)
- Region-based food and transport cost estimates
- Currency conversion for 40+ destinations
- Budget status with savings tips when over budget
- All agents have mock fallback if LLM API unavailable

Next: Phase 5 — Memory & Conversation (MemorySaver, selective re-invocation, human-in-the-loop).